# ROHSA decomposition and HVC source analysis

This notebook provides the end-to-end analysis used to decompose the baseline-subtracted CRAFTS H I cube, reject low-significance Gaussian components, identify and merge spatial sources, derive physical parameters, and inspect fitted spectra. Run the cells from top to bottom after updating the paths and thresholds marked in the configuration sections.

**Coordinate convention:** cube arrays use `(spectral channel, y, x)`, while several ROHSA parameter dictionaries use `(x, y, component)`. Each conversion is documented at the point where it occurs.

**External requirements:** the local CRAFTS FITS cube, a compiled ROHSA executable, `ROHSApy`, and the repository helper scripts are required for a complete run. Data download links are listed in `README.md`.


## 1. Load the H I data cube

Import the numerical, FITS, plotting, and ROHSA interfaces, then load the baseline-subtracted CRAFTS cube. The cell exposes the primary header and the data array for all subsequent steps.


In [ ]:
# Purpose: Load the input FITS cube and initialize the shared analysis objects.

import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from ROHSApy import ROHSA

fitsname = "./data/processed/CRAFTS_cutout_baseline_K.fits"
hdu = fits.open(fitsname)
hdr = hdu[0].header
cube = hdu[0].data


### Inspect the cube dimensions

Print the native array shape in `(spectral channel, y pixel, x pixel)` order.


In [ ]:
# Purpose: Report the cube dimensions before decomposition.

print(hdu[0].data.shape)


### Preview the integrated emission

Sum the cube over the spectral axis to obtain a quick spatial diagnostic. This preview is intended for input verification rather than calibrated science output.


In [ ]:
# Purpose: Display a channel-summed image of the input cube.

plt.figure()
plt.imshow(np.sum(cube,0), origin="lower", cmap="inferno")


## 2. Configure the ROHSA decomposition

Define the number of Gaussian components, regularization strengths, width bounds, noise-channel range, optimizer limits, and output paths. The cell converts the cube to ROHSA's text format and writes the parameter file consumed by the compiled solver.


In [ ]:
# Purpose: Configure ROHSA and generate its cube and parameter files.

filename = "GHIGLS_DFN_Tb_3D.dat"  # ROHSA text representation of the input cube.
fileout = "./baseline/ROHSA_3ngauss_3D_1.1.1.0.dat"  # Gaussian parameter solution written by ROHSA.
filename_noise = ''
n_gauss = 3  # Number of Gaussian components fitted per spectrum.
lambda_amp = 1  # Spatial regularization strength for Gaussian amplitudes.
lambda_mu = 1  # Spatial regularization strength for centroid positions.
lambda_sig = 1  # Spatial regularization strength for velocity dispersions.
lambda_var_sig = 0.  # Regularization strength for dispersion variance.
amp_fact_init = 0.66  # times max amplitude of Gaussians for the fit of the mean spectrum                                                                                                                                                                                                                                                       
sig_init = 4.         # dispersion of Gaussians for the fit of the mean spectrum
lb_sig_init = 1.      # lower limit on sigma for the fit of the mean spectrum
ub_sig_init = 12.     # upper limit on sigma for the fit of the mean spectrum
lb_sig = 1.
ub_sig = 100.                                                                                                                                                                                                                                                                
maxiter_init = 15000  # max iteration for L-BFGS-B alogorithm init mean                                                                                                                                                                                                                                        
maxiter = 800  # Maximum iterations for the full-cube optimization.
noise = ".false."     # if false - STD map computed by ROHSA between lstd and ustd                                                                                                                                                                                                           
lstd = 1  # First channel used by ROHSA for its internal noise estimate.
ustd = 20  # Last channel used by ROHSA for its internal noise estimate.
iprint = -1  # ROHSA optimizer verbosity flag.
iprint_init = -1      # print option init                                              
save_grid = ".false."  # Fortran-style flag controlling multiresolution-grid output.
init_spec =".true."
filename_init_spec = "ngauss_3.txt"


core = ROHSA(cube)            
core.cube2dat(filename=filename)
core.gen_parameters_3D(filename=filename, 
                    fileout=fileout,  
                    n_gauss=n_gauss,
                    lambda_amp=lambda_amp,
                    lambda_mu=lambda_mu,
                    lambda_sig=lambda_sig,
                    lambda_var_sig=lambda_var_sig,
                    amp_fact_init=amp_fact_init,
                    sig_init=sig_init,
                    lb_sig_init=lb_sig_init,
                    ub_sig_init=ub_sig_init,
                    lb_sig=lb_sig,
                    ub_sig=ub_sig,
                    noise=noise,
                    maxiter=maxiter,
                    lstd=lstd,
                    ustd=ustd,
                    iprint_init=iprint_init,
                    iprint=iprint,
                    save_grid=save_grid,
                      init_spec = init_spec,
                   filename_init_spec=filename_init_spec)


### Run the compiled ROHSA solver

Execute the local ROHSA binary with `parameters.txt`. This command requires a compiled ROHSA checkout at `./ROHSA/src/ROHSA`.


In [ ]:
# Purpose: Run the external ROHSA executable using the generated parameter file.

!./ROHSA/src/ROHSA parameters.txt


## 3. Read and inspect the decomposition

Create a fresh ROHSA interface around the input cube before loading the fitted parameters.


In [ ]:
# Purpose: Reinitialize the ROHSA wrapper for post-processing.

core = ROHSA(cube)    


### Reconstruct the fitted cube

Read the Gaussian parameter cube in ROHSA pixel units, reconstruct the model spectrum at every spatial pixel, and separate amplitude, centroid, and dispersion planes.


In [ ]:
# Purpose: Load the fitted Gaussian parameters and reconstruct the model cube.

gaussian = core.read_gaussian("./baseline/ROHSA_3ngauss_3D_1.1.1.0.dat")
print("dim cube = " + str(gaussian.shape))

model = core.return_result_cube(gaussian=gaussian) # Reconstruct the model on the native spectral-channel grid.
# gaussian_phys = core.physical_gaussian(gaussian)
amplitude = gaussian[0::3]
position = gaussian[1::3]
dispersion = gaussian[2::3]

integral = amplitude * dispersion


### Convert Gaussian parameters to physical units

Attach the FITS header so ROHSA can convert centroid and width values from channel units to velocity units. The resulting arrays use kelvin for amplitude and km/s for velocity.


In [ ]:
# Purpose: Convert ROHSA parameters from pixel coordinates to physical units.

# The FITS header provides the spectral-axis calibration.
from astropy.io import fits

core.hdr = hdr

gaussian_phys = core.physical_gaussian(gaussian)

if isinstance(gaussian_phys, np.ndarray):
    print("Conversion succeeded.")
    print("Physical-parameter shape:", gaussian_phys.shape)
    
    # Separate the physical Gaussian parameter planes.
    amplitude_phys = gaussian_phys[0::3]      # Amplitude in brightness-temperature units.
    velocity_phys = gaussian_phys[1::3]       # Gaussian centroid velocity (km/s).
    dispersion_phys = gaussian_phys[2::3]     # Gaussian velocity dispersion (km/s).
    
    print(f"Velocity range: {velocity_phys.min():.2f} to {velocity_phys.max():.2f} km/s")
    print(f"Dispersion range: {dispersion_phys.min():.2f} to {dispersion_phys.max():.2f} km/s")


### Unpack physical parameter maps

Split the interleaved physical parameter cube into amplitude, centroid velocity, and velocity-dispersion arrays for component-level analysis.


In [ ]:
# Purpose: Extract component parameter maps in physical units.

amplitude = gaussian_phys[0::3]
position = gaussian_phys[1::3]
dispersion = gaussian_phys[2::3]

integral = amplitude * dispersion


### Compare input and fitted diagnostics

Plot channel-summed input data, the legacy parameter-sum diagnostic retained from the original analysis, and its normalized difference. Use the reconstructed `model` cube for quantitative residual analysis.


In [ ]:
# Purpose: Display legacy overview diagnostics for the input and fitted products.

plt.figure(figsize=(12,5))
plt.subplot(1,3,1)
plt.imshow(np.sum(cube,0), origin="lower", cmap="inferno")
plt.subplot(1,3,2)
plt.imshow(np.sum(gaussian_phys,0), origin="lower", cmap="inferno")
plt.subplot(1,3,3)
plt.imshow((np.sum(gaussian_phys,0)-np.sum(cube,0))/np.sum(model,0), origin="lower", cmap="inferno")
plt.colorbar()


### Inspect one Gaussian component

Visualize the amplitude, centroid velocity, and velocity dispersion of the selected component index using consistent physical-unit labels.


In [ ]:
# Purpose: Plot the physical parameter maps for one Gaussian component.

i = 0

plt.figure(figsize=(12,5))

plt.subplot(1,3,1)
im1 = plt.imshow(amplitude[i], origin="lower", cmap="inferno")
cbar1 = plt.colorbar(im1, shrink=0.95, pad=0.1, location = 'bottom')  # Place the colorbar below the image.
cbar1.set_label('Brightness Temp. [K]')

plt.subplot(1,3,2)
im2 = plt.imshow(position[i], origin="lower", cmap="coolwarm" )
cbar2 = plt.colorbar(im2, shrink=0.95, pad=0.1, location = 'bottom')  # Place the colorbar below the image.
cbar2.set_label('Velocity [km s$^{-1}$]')

plt.subplot(1,3,3)
im3 = plt.imshow(dispersion[i], origin="lower", cmap="cubehelix")
cbar3 = plt.colorbar(im3, shrink=0.95, pad=0.1, location = 'bottom')  # Place the colorbar below the image.
cbar3.set_label('Dispersion [km s$^{-1}$]')

plt.tight_layout()  # Prevent subplot and label overlap.
plt.show()


### Inspect representative spectra

Compare observed spectra, total ROHSA models, and individual Gaussian components in a 4-by-4 pixel mosaic centered on a user-selected position.


In [ ]:
# Purpose: Plot observed and modeled spectra over a small spatial mosaic.

#Plot mosaic spectra                                                                                                                                                                 
pvalues = np.logspace(-1, 0, len(integral))
pmin = pvalues[0]
pmax = pvalues[-1]

def norm(pval):
    """Normalize a scalar to the configured plotting interval.

        Parameters
        ----------
        pval : float
            Value between the module-level ``pmin`` and ``pmax`` limits.

        Returns
        -------
        float
            Linear normalization of ``pval`` to the interval [0, 1]."""
    return (pval - pmin) / float(pmax - pmin)

ny = 4; nx = 4
# center_y = int(cube.shape[2]/6); center_x = int(cube.shape[1]/6)
# center_y = 20; center_x = 77
center_y = 18; center_x = 24
# center_y = 80; center_x = 230
x = np.arange(cube.shape[0])
cb = "magenta"
cw = "crimson"
fig, axs = plt.subplots(4, 4, sharex=True, sharey=True, figsize=(10.,6.))
fig.subplots_adjust(hspace=0, wspace=0, left=0, right=1, top=1, bottom=0)
for i in np.arange(ny):
    for j in np.arange(nx):
        axs[i][j].step(x, cube[:,center_y+i,center_x+j], color='cornflowerblue', linewidth=2.)
        axs[i][j].plot(x, model[:,center_y+i,center_x+j], linestyle="-", linewidth=2., color="r")
        for k in range(len(integral)):
            axs[i][j].plot(x, core.gauss(x, gaussian[0::3][k][center_y+i,center_x+j],
                                            gaussian[1::3][k][center_y+i,center_x+j], 
                                            gaussian[2::3][k][center_y+i,center_x+j]),
                           linewidth=2., color=plt.cm.inferno(pvalues[k]))
        if j == 0: axs[i][j].set_ylabel(r'T [k]')
        axs[i][j].set_xlabel(r'v [km s$^{-1}$]')
# plt.savefig("plot/" + 'mosaic_spectra_all.png', format='png', bbox_inches='tight', pad_inches=0.02)


## 4. Filter low-significance Gaussian components

Ensure the ROHSA object retains the original FITS header before the filtering pipeline performs any physical-unit conversion.


In [ ]:
# Purpose: Attach the FITS metadata required for physical-unit conversion.

core.hdr = hdr


### Per-pixel noise estimation and S/N filtering

This self-contained pipeline reads ROHSA products, estimates a noise RMS at each sky pixel from line-free channels, applies the S/N threshold independently to every Gaussian component, writes filtered FITS and DAT products, and saves diagnostic plots and statistics.


In [ ]:
# Purpose: Define the complete per-pixel S/N filtering workflow.

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy import ndimage
from mpl_toolkits.axes_grid1 import make_axes_locatable
import os
import warnings
warnings.filterwarnings('ignore')



# 1. Read ROHSA results in pixel and physical units.
def read_rohsa_results(gaussian_file):
    """Read a ROHSA Gaussian solution in pixel and physical units.

        The function relies on the notebook-level ``core`` object, whose FITS header must
        already be set. Parameter planes are returned in ``(x, y, component)`` order.

        Parameters
        ----------
        gaussian_file : str or path-like
            ROHSA ``.dat`` file containing interleaved amplitude, centroid, and
            dispersion planes.

        Returns
        -------
        dict
            Nested pixel- and physical-unit parameter arrays, the component count,
            spatial shape, and the original interleaved arrays."""
    print("\n" + "="*60)
    print("READING ROHSA RESULTS")
    print("="*60)
    
    # Read the native ROHSA parameter cube in pixel units.
    print("Reading Gaussian parameters (pixel units)...")
    gaussian_pixel = core.read_gaussian(gaussian_file)
    
    print(f"Gaussian pixel array shape: {gaussian_pixel.shape}")
    print(f"Data type: {gaussian_pixel.dtype}")
    model = core.return_result_cube(gaussian=gaussian_pixel)
    
    # Convert channel coordinates and widths to physical velocity units.
    print("Converting to physical units...")
    gaussian_physical = core.physical_gaussian(gaussian_pixel)
    
    # Parse the interleaved array shape: (3 * n_components, n_y, n_x).
    n_params_times_comp, n_y, n_x = gaussian_pixel.shape
    n_components = n_params_times_comp // 3
    
    print(f"Spatial dimensions: {n_x} × {n_y}")
    print(f"Number of Gaussian components: {n_components}")
    
    # Extract native pixel-unit parameter maps.
    amplitude_pixel = gaussian_pixel[0::3].transpose(2, 1, 0)  # [n_x, n_y, n_comp]
    position_pixel = gaussian_pixel[1::3].transpose(2, 1, 0)   # [n_x, n_y, n_comp]
    dispersion_pixel = gaussian_pixel[2::3].transpose(2, 1, 0) # [n_x, n_y, n_comp]
    
    # Extract physical-unit parameter maps.
    amplitude_phys = gaussian_physical[0::3].transpose(2, 1, 0)  # [n_x, n_y, n_comp]
    position_phys = gaussian_physical[1::3].transpose(2, 1, 0)   # [n_x, n_y, n_comp]
    dispersion_phys = gaussian_physical[2::3].transpose(2, 1, 0) # [n_x, n_y, n_comp]
    
    # Replace non-finite values and enforce non-negative dispersions.
    amplitude_pixel = np.nan_to_num(amplitude_pixel)
    position_pixel = np.nan_to_num(position_pixel)
    dispersion_pixel = np.nan_to_num(np.abs(dispersion_pixel))
    
    amplitude_phys = np.nan_to_num(amplitude_phys)
    position_phys = np.nan_to_num(position_phys)
    dispersion_phys = np.nan_to_num(np.abs(dispersion_phys))
    
    print(f"\nPixel units statistics:")
    for comp in range(n_components):
        amp_comp = amplitude_pixel[:, :, comp]
        amp_nonzero = amp_comp[amp_comp > 0]
        if len(amp_nonzero) > 0:
            print(f"  Component {comp+1} amplitude (pixel): range [{np.min(amp_nonzero):.4f}, {np.max(amp_comp):.4f}]")
    
    print(f"\nPhysical units statistics:")
    for comp in range(n_components):
        amp_comp = amplitude_phys[:, :, comp]
        amp_nonzero = amp_comp[amp_comp > 0]
        if len(amp_nonzero) > 0:
            print(f"  Component {comp+1} amplitude (K): range [{np.min(amp_nonzero):.4f}, {np.max(amp_comp):.4f}]")
    
    return {
        'pixel': {
            'amplitude': amplitude_pixel,
            'position': position_pixel,
            'dispersion': dispersion_pixel,
        },
        'physical': {
            'amplitude': amplitude_phys,
            'position': position_phys,
            'dispersion': dispersion_phys,
        },
        'n_components': n_components,
        'shape': (n_x, n_y),
        'original_pixel': gaussian_pixel,
        'original_physical': gaussian_physical
    }

# 2. Read the original FITS cube.
def read_original_fits(fits_file):
    """Read the original H I cube and derive its spectral coordinate.

        Parameters
        ----------
        fits_file : str or path-like
            Input FITS cube in ``(channel, y, x)`` order.

        Returns
        -------
        cube : numpy.ndarray or None
            Primary data cube, or ``None`` if the file cannot be read.
        header : astropy.io.fits.Header or None
            Primary FITS header.
        velocities : numpy.ndarray or None
            Linear spectral coordinate derived from ``CRVAL3`` and ``CDELT3``.
            Values are assumed to use the velocity unit encoded by the FITS header."""
    print("\n" + "="*60)
    print("READING ORIGINAL FITS")
    print("="*60)
    
    try:
        hdu = fits.open(fits_file)[0]
        cube = hdu.data
        header = hdu.header
        
        print(f"Original cube shape: {cube.shape}")
        print(f"Data range: [{np.min(cube):.4f}, {np.max(cube):.4f}]")
        
        # Derive a linear spectral coordinate when the required WCS keywords exist.
        if 'CRVAL3' in header and 'CDELT3' in header and 'NAXIS3' in header:
            v0 = header['CRVAL3']
            dv = header['CDELT3']
            n_channels = header['NAXIS3']
            velocities = v0 + np.arange(n_channels) * dv
            print(f"Velocity range: [{velocities[0]:.2f}, {velocities[-1]:.2f}] km/s")
        else:
            velocities = None
            print("No velocity information in header")
        
        return cube, header, velocities
        
    except Exception as e:
        print(f"Error reading FITS file: {e}")
        return None, None, None

# 3. Estimate the RMS noise independently at every spatial pixel.
def calculate_per_pixel_noise(original_cube, output_dir, noise_channels=None, velocities=None):
    """Estimate a spatial RMS-noise map from line-free channels.

        Two channel intervals are concatenated before computing the standard deviation
        along the spectral axis. When no intervals are supplied, the first and last
        10 percent of the cube are used.

        Parameters
        ----------
        original_cube : numpy.ndarray
            H I brightness-temperature cube in ``(channel, y, x)`` order.
        output_dir : str or path-like
            Directory in which to save the noise-distribution histogram.
        noise_channels : sequence of four int, optional
            ``(start_1, stop_1, start_2, stop_2)`` channel bounds; stop indices are
            exclusive.
        velocities : numpy.ndarray, optional
            Spectral coordinate retained for interface compatibility. It is not used
            by the current channel-index implementation.

        Returns
        -------
        noise_rms_map : numpy.ndarray or None
            Per-pixel RMS map in ``(y, x)`` order.
        global_noise_rms : float or None
            Mean of the per-pixel RMS values."""
    print("\n" + "="*60)
    print("CALCULATING PER-PIXEL NOISE FROM DATA")
    print("="*60)
    
    if original_cube is None:
        print("No original cube data available")
        return None, None
    
    n_channels, n_y, n_x = original_cube.shape
    
    if noise_channels is None:
        # Default to the first and last 10 percent of channels as line-free data.
        n_noise = max(10, int(n_channels * 0.1))
        noise_channels_indices = [0, n_noise, n_channels - n_noise, n_channels]
        print(f"Auto-selected noise channels: indices {noise_channels_indices}")
    else:
        noise_channels_indices = noise_channels
        print(f"Using specified noise channels: {noise_channels_indices}")
    
    # Concatenate the two line-free channel intervals.
    start1, end1, start2, end2 = noise_channels_indices
    noise_data1 = original_cube[start1:end1, :, :]
    noise_data2 = original_cube[start2:end2, :, :]
    noise_data = np.concatenate([noise_data1, noise_data2], axis=0)
    
    print(f"Using {len(noise_data)} channels for noise calculation")
    
    # Compute the spectral standard deviation at every sky pixel.
    noise_rms_map = np.std(noise_data, axis=0)  # [n_y, n_x]
    global_noise_rms = np.mean(noise_rms_map)
    
    print(f"\nPer-pixel noise statistics (K):")
    print(f"  Global mean RMS: {global_noise_rms:.4f}")
    print(f"  RMS range: [{np.min(noise_rms_map):.4f}, {np.max(noise_rms_map):.4f}]")
    print(f"  RMS median: {np.median(noise_rms_map):.4f}")
    
    # Summarize the spatial distribution of RMS values.
    plt.figure(figsize=(10, 6))
    plt.hist(noise_rms_map.flatten(), bins=50, alpha=0.7, color='steelblue', edgecolor='black')
    plt.axvline(global_noise_rms, color='red', linestyle='--', linewidth=2, 
                label=f'Mean: {global_noise_rms:.4f} K')
    plt.axvline(np.median(noise_rms_map), color='green', linestyle='--', linewidth=2,
                label=f'Median: {np.median(noise_rms_map):.4f} K')
    plt.xlabel('Noise RMS [K]', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.title('Distribution of Per-Pixel Noise RMS', fontsize=14)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    
    # Save the noise-distribution diagnostic.
    hist_file = os.path.join(output_dir, 'noise_distribution.png')
    plt.savefig(hist_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Noise distribution histogram saved to: {hist_file}")
    
    return noise_rms_map, global_noise_rms

# 4. Apply the amplitude S/N threshold independently to each component.
def filter_each_component_with_per_pixel_noise(rohsa_results, noise_rms_map, snr_threshold=2.0):
    """Apply an independent per-pixel S/N cut to each Gaussian component.

        Signal-to-noise is defined as physical amplitude in kelvin divided by the
        local RMS noise. Rejected pixels are set to zero in both pixel-unit and
        physical-unit parameter arrays.

        Parameters
        ----------
        rohsa_results : dict
            Output from :func:`read_rohsa_results`.
        noise_rms_map : numpy.ndarray
            Local RMS noise in ``(y, x)`` order.
        snr_threshold : float, default=2.0
            Minimum amplitude-to-noise ratio retained in each component.

        Returns
        -------
        filtered_results : dict
            Copy of the ROHSA parameter structure with rejected entries zeroed.
        masks : numpy.ndarray or None
            Boolean retention masks in ``(x, y, component)`` order.
        snr_maps : numpy.ndarray or None
            Component-level S/N maps with the same ordering as ``masks``."""
    print("\n" + "="*60)
    print("FILTERING EACH COMPONENT WITH PER-PIXEL NOISE")
    print("="*60)
    print(f"SNR threshold: {snr_threshold}σ")
    
    if noise_rms_map is None:
        print("No noise map available, skipping filtering")
        return rohsa_results, None, None
    
    # Use physical amplitudes in kelvin for the S/N calculation.
    amplitude_phys = rohsa_results['physical']['amplitude'].copy()  # Shape: (n_x, n_y, n_components); unit: K.
    n_components = rohsa_results['n_components']
    n_x, n_y, _ = amplitude_phys.shape
    
    # Add a component axis and transpose the RMS map to (x, y, 1).
    noise_expanded = noise_rms_map.T[:, :, np.newaxis]  # [n_x, n_y, 1]
    
    # Compute one S/N value per pixel and Gaussian component.
    with np.errstate(divide='ignore', invalid='ignore'):
        snr_maps = amplitude_phys / (noise_expanded + 1e-10)
        snr_maps = np.nan_to_num(snr_maps)
    
    print(f"\nSNR statistics before filtering:")
    for comp in range(n_components):
        comp_snr = snr_maps[:, :, comp]
        valid_snr = comp_snr[amplitude_phys[:, :, comp] > 0]
        if len(valid_snr) > 0:
            print(f"  Component {comp+1}: SNR range [{np.min(valid_snr):.2f}, {np.max(valid_snr):.2f}], "
                  f"mean {np.mean(valid_snr):.2f}")
    
    # Retain entries that meet the physical-amplitude S/N threshold.
    masks = snr_maps >= snr_threshold  # [n_x, n_y, n_comp]
    
    print(f"\nPer component statistics after filtering:")
    for comp in range(n_components):
        orig_pixels = np.sum(amplitude_phys[:, :, comp] > 0)
        keep_pixels = np.sum(masks[:, :, comp])
        
        if orig_pixels > 0:
            retention = 100 * keep_pixels / orig_pixels
        else:
            retention = 0
            
        percent_total = 100 * keep_pixels / (n_x * n_y)
        print(f"  Component {comp+1}:")
        print(f"    Original non-zero: {orig_pixels}")
        print(f"    Kept: {keep_pixels} ({percent_total:.2f}% of total, {retention:.2f}% of original)")
        
        # Report the S/N range among retained pixels.
        kept_snr = snr_maps[:, :, comp][masks[:, :, comp]]
        if len(kept_snr) > 0:
            print(f"    Kept SNR range: [{np.min(kept_snr):.2f}, {np.max(kept_snr):.2f}]")
    
    # Copy both parameter representations before applying the masks.
    filtered_results = {
        'pixel': {
            'amplitude': rohsa_results['pixel']['amplitude'].copy(),
            'position': rohsa_results['pixel']['position'].copy(),
            'dispersion': rohsa_results['pixel']['dispersion'].copy(),
        },
        'physical': {
            'amplitude': rohsa_results['physical']['amplitude'].copy(),
            'position': rohsa_results['physical']['position'].copy(),
            'dispersion': rohsa_results['physical']['dispersion'].copy(),
        },
        'n_components': n_components,
        'shape': (n_x, n_y)
    }
    
    # Apply identical masks to native and physical parameter arrays.
    for comp in range(n_components):
        comp_mask = masks[:, :, comp]
        
        # Native ROHSA pixel units.
        filtered_results['pixel']['amplitude'][~comp_mask, comp] = 0
        filtered_results['pixel']['position'][~comp_mask, comp] = 0
        filtered_results['pixel']['dispersion'][~comp_mask, comp] = 0
        
        # Physical units used for scientific interpretation.
        filtered_results['physical']['amplitude'][~comp_mask, comp] = 0
        filtered_results['physical']['position'][~comp_mask, comp] = 0
        filtered_results['physical']['dispersion'][~comp_mask, comp] = 0
    
    return filtered_results, masks, snr_maps

# 5. Save filtered physical parameters to FITS.
def save_filtered_rohsa_fits(filtered_results, output_file):
    """Write filtered physical Gaussian parameters to a FITS cube.

        Parameters
        ----------
        filtered_results : dict
            Filtered result structure produced by the S/N filter.
        output_file : str or path-like
            Destination FITS path. Existing files are overwritten.

        Returns
        -------
        bool
            ``True`` on success and ``False`` when writing fails."""
    try:
        n_components = filtered_results['n_components']
        n_x, n_y = filtered_results['shape']
        
        # Reassemble interleaved parameter planes in (3 * component, y, x) order.
        combined = np.zeros((3*n_components, n_y, n_x))
        
        for comp in range(n_components):
            # Store amplitude, centroid velocity, and dispersion in physical units.
            combined[3*comp] = filtered_results['physical']['amplitude'][:, :, comp].T
            combined[3*comp + 1] = filtered_results['physical']['position'][:, :, comp].T
            combined[3*comp + 2] = filtered_results['physical']['dispersion'][:, :, comp].T
        
        # Overwrite the destination to make reruns deterministic.
        hdu = fits.PrimaryHDU(combined)
        hdu.writeto(output_file, overwrite=True)
        
        print(f"\nFiltered ROHSA results saved to FITS (physical units): {output_file}")
        return True
    except Exception as e:
        print(f"Error saving filtered results to FITS: {e}")
        return False

# 6. Save filtered native parameters in ROHSA DAT format.
def save_filtered_rohsa_dat(filtered_results, original_dat_file, output_file):
    """Write filtered pixel-unit parameters in native ROHSA DAT format.

        The original header is copied when available. Data rows retain ROHSA's
        ``y x amplitude centroid dispersion`` ordering and fixed-width formatting.

        Parameters
        ----------
        filtered_results : dict
            Filtered result structure produced by the S/N filter.
        original_dat_file : str or path-like
            Unfiltered ROHSA DAT file used as the header template.
        output_file : str or path-like
            Destination DAT path.

        Returns
        -------
        bool
            ``True`` on success and ``False`` when writing fails."""
    print("\n" + "="*60)
    print("SAVING FILTERED RESULTS TO .DAT FORMAT (PIXEL UNITS)")
    print("="*60)
    
    try:
        n_components = filtered_results['n_components']
        n_x, n_y = filtered_results['shape']  # Spatial dimensions are stored as (x, y).
        
        print(f"  Spatial grid: X({n_x}) × Y({n_y})")
        print(f"  Components: {n_components}")
        print(f"  DAT file format: Y X AMP POS DISP (Y first, X second)")
        print(f"  Units: Pixel units (matching original ROHSA format)")
        
        # Count retained entries using native amplitude values.
        total_entries = n_x * n_y * n_components
        nonzero_entries = np.sum(filtered_results['pixel']['amplitude'] != 0)
        print(f"  Non-zero entries: {nonzero_entries} ({100 * nonzero_entries / total_entries:.2f}%)")
        
        # Preserve the original ROHSA header when possible.
        header_lines = []
        try:
            with open(original_dat_file, 'r') as f:
                for i in range(27):  # Preserve the 27-line ROHSA header.
                    line = f.readline()
                    header_lines.append(line)
            print(f"  Read {len(header_lines)} header lines from original file")
        except Exception as e:
            print(f"  Warning: Could not read original file header: {e}")
            print("  Using default header")
            header_lines = [
                f"# ROHSA filtered results (pixel units)\n",
                f"# Number of components: {n_components}\n",
                f"# Grid size: X={n_x}, Y={n_y}\n",
                f"# SNR threshold applied: Yes\n",
                f"# Non-zero entries: {nonzero_entries} / {total_entries}\n",
                f"#\n",
                f"# Column 1: Y (pixel)\n",
                f"# Column 2: X (pixel)\n",
                f"# Column 3: Amplitude (pixel units)\n",
                f"# Column 4: Position (pixel units)\n",
                f"# Column 5: Dispersion (pixel units)\n",
                f"#\n"
            ]
        
        # Write the filtered native-unit DAT product.
        with open(output_file, 'w') as f:
            # Copy the source header before writing parameter rows.
            f.writelines(header_lines)
            
            # Preserve ROHSA row ordering: y coordinate before x coordinate.
            line_count = 0
            for j in range(n_y):  # Outer loop over y.
                for i in range(n_x):  # Inner loop over x.
                    for comp in range(n_components):
                        # Read values from the native-unit arrays.
                        amp = filtered_results['pixel']['amplitude'][i, j, comp]
                        pos = filtered_results['pixel']['position'][i, j, comp]
                        dis = filtered_results['pixel']['dispersion'][i, j, comp]
                        
                        # Match the fixed-width numeric formatting of the original file.
                        line = f"    {j:4d}    {i:4d}    {amp:20.16f}    {pos:20.16f}    {dis:20.16f}\n"
                        f.write(line)
                        line_count += 1
            
            print(f"  Total lines written: {line_count}")
            print(f"  Data format: Y(4d) X(4d) AMP(20.16f) POS(20.16f) DISP(20.16f)")
        
        print(f"  DAT file saved to: {output_file}")
        
        # Preview a small number of data rows as a write check.
        print(f"\n  First few lines of saved file:")
        with open(output_file, 'r') as f:
            for i, line in enumerate(f):
                if i >= 5:  # Preview five data rows.
                    break
                if i >= len(header_lines):  # Skip header lines.
                    print(f"    {line.strip()}")
        
        return True
        
    except Exception as e:
        print(f"Error saving filtered results to DAT: {e}")
        import traceback
        traceback.print_exc()
        return False

# 7. Save both physical FITS and native DAT representations.
def save_filtered_rohsa_all(filtered_results, original_dat_file, output_base):
    """Save filtered ROHSA parameters in visualization and solver formats.

        Parameters
        ----------
        filtered_results : dict
            Filtered result structure produced by the S/N filter.
        original_dat_file : str or path-like
            Original DAT file used to preserve the ROHSA header.
        output_base : str or path-like
            Output path without an extension.

        Returns
        -------
        bool
            ``True`` after dispatching both output writers."""
    print("\n" + "="*60)
    print("SAVING FILTERED ROHSA RESULTS")
    print("="*60)
    
    # FITS is intended for physical-unit visualization and analysis.
    fits_file = output_base + '.fits'
    save_filtered_rohsa_fits(filtered_results, fits_file)
    
    # DAT preserves the native format expected by ROHSA utilities.
    dat_file = output_base + '.dat'
    save_filtered_rohsa_dat(filtered_results, original_dat_file, dat_file)
    
    return True

# 8. Compare original and filtered physical parameter maps.
def plot_component_comparison(original_results, filtered_results, noise_map, comp_index, output_dir, snr_threshold):
    """Compare original and S/N-filtered maps for one component.

        Parameters
        ----------
        original_results, filtered_results : dict
            ROHSA parameter structures before and after filtering.
        noise_map : numpy.ndarray
            Local RMS map; accepted for a consistent diagnostic interface.
        comp_index : int
            Zero-based Gaussian component index.
        output_dir : str or path-like
            Directory for the saved comparison figure.
        snr_threshold : float
            Threshold reported in the figure title."""
    # Select one component and transpose maps to image order (y, x).
    orig_amp = original_results['physical']['amplitude'][:, :, comp_index].T  # [n_y, n_x]
    orig_pos = original_results['physical']['position'][:, :, comp_index].T
    orig_dis = original_results['physical']['dispersion'][:, :, comp_index].T
    
    filt_amp = filtered_results['physical']['amplitude'][:, :, comp_index].T
    filt_pos = filtered_results['physical']['position'][:, :, comp_index].T
    filt_dis = filtered_results['physical']['dispersion'][:, :, comp_index].T
    
    print(f"\nComponent {comp_index+1} plot ranges (physical units):")
    
    # Derive shared color limits from valid nonzero values.
    amp_mask = orig_amp > 0
    if np.any(amp_mask):
        amp_vmin = np.min(orig_amp[amp_mask])
        amp_vmax = np.max(orig_amp)
        print(f"  Amplitude range: [{amp_vmin:.4f}, {amp_vmax:.4f}] K")
    else:
        amp_vmin, amp_vmax = 0, 1
        print("  No valid amplitude values")
    
    pos_mask = (orig_pos != 0) & (~np.isnan(orig_pos))
    if np.any(pos_mask):
        pos_vmin = np.min(orig_pos[pos_mask])
        pos_vmax = np.max(orig_pos)
        print(f"  Position range: [{pos_vmin:.4f}, {pos_vmax:.4f}] km/s")
    else:
        pos_vmin, pos_vmax = -10, 10
        print("  No valid position values")
    
    dis_mask = orig_dis > 0
    if np.any(dis_mask):
        dis_vmin = np.min(orig_dis[dis_mask])
        dis_vmax = np.max(orig_dis)
        print(f"  Dispersion range: [{dis_vmin:.4f}, {dis_vmax:.4f}] km/s")
    else:
        dis_vmin, dis_vmax = 0, 10
        print("  No valid dispersion values")
    
    # Arrange original and filtered maps in a 2-by-3 grid.
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Top row: unfiltered component parameters.
    # Original amplitude.
    im1 = axes[0, 0].imshow(orig_amp, origin='lower', cmap='inferno', vmin=amp_vmin, vmax=amp_vmax)
    axes[0, 0].set_title(f'Component {comp_index+1} - Original Amplitude', fontsize=12)
    cbar1 = plt.colorbar(im1, ax=axes[0, 0], shrink=0.8, pad=0.05, location='bottom')
    cbar1.set_label('Brightness Temp. [K]', fontsize=10)
    
    # Original centroid velocity.
    im2 = axes[0, 1].imshow(orig_pos, origin='lower', cmap='coolwarm', vmin=pos_vmin, vmax=pos_vmax)
    axes[0, 1].set_title(f'Component {comp_index+1} - Original Position', fontsize=12)
    cbar2 = plt.colorbar(im2, ax=axes[0, 1], shrink=0.8, pad=0.05, location='bottom')
    cbar2.set_label('Velocity [km s$^{-1}$]', fontsize=10)
    
    # Original velocity dispersion.
    im3 = axes[0, 2].imshow(orig_dis, origin='lower', cmap='cubehelix', vmin=dis_vmin, vmax=dis_vmax)
    axes[0, 2].set_title(f'Component {comp_index+1} - Original Dispersion', fontsize=12)
    cbar3 = plt.colorbar(im3, ax=axes[0, 2], shrink=0.8, pad=0.05, location='bottom')
    cbar3.set_label('Dispersion [km s$^{-1}$]', fontsize=10)
    
    # Bottom row: filtered parameters with identical color limits.
    # Filtered amplitude.
    im4 = axes[1, 0].imshow(filt_amp, origin='lower', cmap='inferno', vmin=amp_vmin, vmax=amp_vmax)
    axes[1, 0].set_title(f'Component {comp_index+1} - Filtered Amplitude', fontsize=12)
    cbar4 = plt.colorbar(im4, ax=axes[1, 0], shrink=0.8, pad=0.05, location='bottom')
    cbar4.set_label('Brightness Temp. [K]', fontsize=10)
    
    # Filtered centroid velocity.
    im5 = axes[1, 1].imshow(filt_pos, origin='lower', cmap='coolwarm', vmin=pos_vmin, vmax=pos_vmax)
    axes[1, 1].set_title(f'Component {comp_index+1} - Filtered Position', fontsize=12)
    cbar5 = plt.colorbar(im5, ax=axes[1, 1], shrink=0.8, pad=0.05, location='bottom')
    cbar5.set_label('Velocity [km s$^{-1}$]', fontsize=10)
    
    # Filtered velocity dispersion.
    im6 = axes[1, 2].imshow(filt_dis, origin='lower', cmap='cubehelix', vmin=dis_vmin, vmax=dis_vmax)
    axes[1, 2].set_title(f'Component {comp_index+1} - Filtered Dispersion', fontsize=12)
    cbar6 = plt.colorbar(im6, ax=axes[1, 2], shrink=0.8, pad=0.05, location='bottom')
    cbar6.set_label('Dispersion [km s$^{-1}$]', fontsize=10)
    
    # Record the component index and applied S/N threshold.
    plt.suptitle(f'Component {comp_index+1}: Original vs Filtered (Per-pixel {snr_threshold}σ threshold)', 
                fontsize=14, y=1.02)
    
    plt.tight_layout()
    
    # Save and close the figure to release memory during batch runs.
    comp_file = os.path.join(output_dir, f'component{comp_index+1}_comparison.png')
    plt.savefig(comp_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Component {comp_index+1} comparison plot saved to: {comp_file}")


# 9. Plot all component masks and S/N maps.
def plot_masks_and_snr(masks, snr_maps, noise_map, output_dir, snr_threshold):
    """Plot component retention masks and S/N maps.

        Parameters
        ----------
        masks : numpy.ndarray
            Boolean masks in ``(x, y, component)`` order.
        snr_maps : numpy.ndarray
            Signal-to-noise maps with the same shape as ``masks``.
        noise_map : numpy.ndarray
            Local RMS map retained for interface consistency.
        output_dir : str or path-like
            Directory for PNG outputs.
        snr_threshold : float
            S/N threshold shown on the diagnostics."""
    if masks is None or snr_maps is None:
        print("No masks or SNR maps to plot")
        return
    
    n_components = masks.shape[2]
    n_x, n_y, _ = masks.shape
    
    # Display the retained-pixel mask for each component.
    fig, axes = plt.subplots(1, n_components, figsize=(5*n_components, 4))
    if n_components == 1:
        axes = [axes]
    
    for comp in range(n_components):
        im = axes[comp].imshow(masks[:, :, comp].T, origin='lower', cmap='gray', aspect='auto')
        axes[comp].set_title(f'Component {comp+1} Mask', fontsize=12)
        axes[comp].set_xlabel('X pixel')
        axes[comp].set_ylabel('Y pixel')
        divider = make_axes_locatable(axes[comp])
        cax = divider.append_axes("right", size="5%", pad=0.05)
        cbar = plt.colorbar(im, cax=cax)
        cbar.set_label('Keep (1) / Remove (0)')
        
        # Annotate each panel with retained-pixel statistics.
        keep_pixels = np.sum(masks[:, :, comp])
        total_pixels = n_x * n_y
        axes[comp].text(0.02, 0.98, f'Keep: {keep_pixels} ({100*keep_pixels/total_pixels:.1f}%)', 
                       transform=axes[comp].transAxes, fontsize=10,
                       verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.suptitle(f'Component Masks (Per-pixel {snr_threshold}σ threshold)', fontsize=14)
    plt.tight_layout()
    
    mask_file = os.path.join(output_dir, 'all_masks.png')
    plt.savefig(mask_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"All masks plot saved to: {mask_file}")
    
    # Display component-level signal-to-noise maps.
    fig, axes = plt.subplots(1, n_components, figsize=(5*n_components, 4))
    if n_components == 1:
        axes = [axes]
    
    # Use robust shared S/N limits across all components.
    valid_snr = snr_maps[snr_maps > 0]
    if len(valid_snr) > 0:
        snr_max = np.percentile(valid_snr, 95)
    else:
        snr_max = 10
    snr_min = 0
    
    for comp in range(n_components):
        im = axes[comp].imshow(snr_maps[:, :, comp].T, origin='lower', cmap='hot', 
                              vmin=snr_min, vmax=snr_max, aspect='auto')
        axes[comp].set_title(f'Component {comp+1} SNR', fontsize=12)
        axes[comp].set_xlabel('X pixel')
        axes[comp].set_ylabel('Y pixel')
        divider = make_axes_locatable(axes[comp])
        cax = divider.append_axes("right", size="5%", pad=0.05)
        cbar = plt.colorbar(im, cax=cax)
        cbar.set_label('SNR')
        
        # Mark the applied threshold in the panel annotation.
        if masks is not None:
            axes[comp].contour(masks[:, :, comp].T, levels=[0.5], colors='cyan', linewidths=0.5, alpha=0.5)
    
    plt.suptitle(f'Component SNR Maps (Per-pixel noise, threshold={snr_threshold}σ)', fontsize=14)
    plt.tight_layout()
    
    snr_file = os.path.join(output_dir, 'all_snr_maps.png')
    plt.savefig(snr_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"All SNR maps saved to: {snr_file}")

# 10. Plot the spatial RMS-noise map.
def plot_noise_map(noise_map, output_dir):
    """Save the spatial RMS-noise map.

        Parameters
        ----------
        noise_map : numpy.ndarray
            Per-pixel RMS map in ``(y, x)`` order.
        output_dir : str or path-like
            Directory for the output PNG."""
    if noise_map is None:
        print("No noise map to plot")
        return
    
    plt.figure(figsize=(10, 8))
    im = plt.imshow(noise_map, origin='lower', cmap='hot', aspect='auto')
    plt.colorbar(im, label='Noise RMS [K]', shrink=0.8)
    plt.title('Per-Pixel Noise RMS Map', fontsize=14)
    plt.xlabel('X pixel', fontsize=12)
    plt.ylabel('Y pixel', fontsize=12)
    plt.tight_layout()
    
    noise_file = os.path.join(output_dir, 'noise_map.png')
    plt.savefig(noise_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Noise map saved to: {noise_file}")

# 11. Save component-level filtering statistics.
def save_statistics(original_results, filtered_results, masks, noise_map, snr_threshold, output_dir):
    """Write component-level filtering statistics to a text report.

        Parameters
        ----------
        original_results, filtered_results : dict
            ROHSA parameter structures before and after filtering.
        masks : numpy.ndarray
            Boolean component-retention masks.
        noise_map : numpy.ndarray
            Per-pixel RMS map.
        snr_threshold : float
            Applied amplitude-to-noise threshold.
        output_dir : str or path-like
            Directory for the statistics report."""
    n_components = original_results['n_components']
    n_x, n_y = original_results['shape']
    total_pixels = n_x * n_y
    
    stats_file = os.path.join(output_dir, 'statistics.txt')
    
    with open(stats_file, 'w') as f:
        f.write("="*60 + "\n")
        f.write("FILTERING STATISTICS\n")
        f.write("="*60 + "\n\n")
        
        f.write(f"Total pixels: {total_pixels}\n")
        f.write(f"SNR threshold: {snr_threshold}σ (per-pixel)\n")
        if noise_map is not None:
            f.write(f"Noise RMS range: [{np.min(noise_map):.4f}, {np.max(noise_map):.4f}] K\n")
            f.write(f"Noise RMS mean: {np.mean(noise_map):.4f} K\n")
            f.write(f"Noise RMS median: {np.median(noise_map):.4f} K\n\n")
        
        f.write("-"*40 + "\n")
        f.write("PER COMPONENT STATISTICS\n")
        f.write("-"*40 + "\n\n")
        
        for comp in range(n_components):
            orig_amp = original_results['physical']['amplitude'][:, :, comp]
            filt_amp = filtered_results['physical']['amplitude'][:, :, comp]
            
            orig_nonzero = np.sum(orig_amp > 0)
            filt_nonzero = np.sum(filt_amp > 0)
            
            f.write(f"Component {comp+1}:\n")
            f.write(f"  Original non-zero pixels: {orig_nonzero} ({100*orig_nonzero/total_pixels:.2f}% of total)\n")
            f.write(f"  Filtered non-zero pixels: {filt_nonzero} ({100*filt_nonzero/total_pixels:.2f}% of total)\n")
            
            if orig_nonzero > 0:
                f.write(f"  Retention rate: {100*filt_nonzero/orig_nonzero:.2f}%\n")
            
            if filt_nonzero > 0:
                f.write(f"\n  Amplitude (K):\n")
                f.write(f"    Original range: [{np.min(orig_amp[orig_amp>0]):.4f}, {np.max(orig_amp):.4f}]\n")
                f.write(f"    Filtered range: [{np.min(filt_amp[filt_amp>0]):.4f}, {np.max(filt_amp):.4f}]\n")
                f.write(f"    Original mean: {np.mean(orig_amp[orig_amp>0]):.4f}\n")
                f.write(f"    Filtered mean: {np.mean(filt_amp[filt_amp>0]):.4f}\n")
                
                # Summarize retained centroid velocities.
                orig_pos = original_results['physical']['position'][:, :, comp]
                filt_pos = filtered_results['physical']['position'][:, :, comp]
                f.write(f"\n  Position (km/s):\n")
                f.write(f"    Original range: [{np.min(orig_pos[orig_pos!=0]):.4f}, {np.max(orig_pos):.4f}]\n")
                f.write(f"    Filtered range: [{np.min(filt_pos[filt_pos!=0]):.4f}, {np.max(filt_pos):.4f}]\n")
                
                # Summarize retained velocity dispersions.
                orig_dis = original_results['physical']['dispersion'][:, :, comp]
                filt_dis = filtered_results['physical']['dispersion'][:, :, comp]
                f.write(f"\n  Dispersion (km/s):\n")
                f.write(f"    Original range: [{np.min(orig_dis[orig_dis>0]):.4f}, {np.max(orig_dis):.4f}]\n")
                f.write(f"    Filtered range: [{np.min(filt_dis[filt_dis>0]):.4f}, {np.max(filt_dis):.4f}]\n")
            
            # Measure local-noise statistics over retained pixels.
            if masks is not None and noise_map is not None:
                kept_mask = masks[:, :, comp]
                kept_noise = noise_map.T[kept_mask]
                if len(kept_noise) > 0:
                    f.write(f"\n  Noise in kept pixels (K):\n")
                    f.write(f"    Mean: {np.mean(kept_noise):.4f}\n")
                    f.write(f"    Std: {np.std(kept_noise):.4f}\n")
                    f.write(f"    Range: [{np.min(kept_noise):.4f}, {np.max(kept_noise):.4f}]\n")
            
            f.write("\n" + "-"*30 + "\n\n")
    
    print(f"Statistics saved to: {stats_file}")

# 12. Orchestrate the complete S/N filtering stage.
def main(gaussian_file, fits_file, output_dir, noise_channels=None, snr_threshold=2.0):
    """Run the complete per-pixel S/N filtering workflow.

        Parameters
        ----------
        gaussian_file : str or path-like
            Input ROHSA Gaussian solution in DAT format.
        fits_file : str or path-like
            Original H I cube used to derive the local RMS-noise map.
        output_dir : str or path-like
            Directory for filtered products, diagnostics, and statistics.
        noise_channels : sequence of four int, optional
            Two half-open channel intervals used for noise estimation.
        snr_threshold : float, default=2.0
            Minimum component amplitude-to-noise ratio.

        Returns
        -------
        filtered_results : dict
            Filtered native- and physical-unit Gaussian parameter arrays.
        masks : numpy.ndarray
            Boolean retention masks in ``(x, y, component)`` order."""
    
    print("\n" + "="*60)
    print("PER-PIXEL NOISE COMPONENT-WISE SNR FILTERING")
    print("="*60)
    print(f"ROHSA file: {gaussian_file}")
    print(f"FITS file: {fits_file}")
    print(f"Output directory: {output_dir}")
    print(f"SNR threshold: {snr_threshold}σ")
    
    # Create output directories before any file-producing operation.
    os.makedirs(output_dir, exist_ok=True)
    print(f"Created output directory: {output_dir}")
    
    # Read native and physical ROHSA parameter representations.
    original_results = read_rohsa_results(gaussian_file)
    n_components = original_results['n_components']
    
    # Load the original cube used for local-noise estimation.
    original_cube, header, velocities = read_original_fits(fits_file)
    
    # Estimate one RMS value per spatial pixel.
    noise_map, global_noise = calculate_per_pixel_noise(original_cube, output_dir, noise_channels, velocities)
    
    # Apply the S/N criterion independently to every component.
    filtered_results, masks, snr_maps = filter_each_component_with_per_pixel_noise(
        original_results, 
        noise_map, 
        snr_threshold
    )
    
    # Write the filtered FITS and DAT products.
    output_base = os.path.join(output_dir, 'filtered_rohsa')
    save_filtered_rohsa_all(filtered_results, gaussian_file, output_base)
    
    # Save original-versus-filtered diagnostics for every component.
    for comp in range(n_components):
        plot_component_comparison(original_results, filtered_results, noise_map, comp, output_dir, snr_threshold)
    
    # Save aggregate mask and signal-to-noise diagnostics.
    plot_masks_and_snr(masks, snr_maps, noise_map, output_dir, snr_threshold)
    
    # Save the spatial RMS map.
    plot_noise_map(noise_map, output_dir)
    
    # Write a machine-readable audit summary of the filtering stage.
    save_statistics(original_results, filtered_results, masks, noise_map, snr_threshold, output_dir)
    
    print("\n" + "="*60)
    print("PROCESSING COMPLETED")
    print("="*60)
    print(f"All output files saved to: {output_dir}")
    print(f"  - Filtered FITS (physical units): filtered_rohsa.fits")
    print(f"  - Filtered DAT (pixel units): filtered_rohsa.dat")
    for comp in range(n_components):
        print(f"  - Component {comp+1} plot: component{comp+1}_comparison.png")
    print(f"  - All masks: all_masks.png")
    print(f"  - All SNR maps: all_snr_maps.png")
    print(f"  - Noise map: noise_map.png")
    print(f"  - Noise distribution: noise_distribution.png")
    print(f"  - Statistics: statistics.txt")
    
    return filtered_results, masks

# 13. Configure and run the S/N filtering stage.
if __name__ == "__main__":
    # Update these paths for the local data layout.
    GAUSSIAN_FILE = "./baseline/ROHSA_3ngauss_3D_1.1.1.0.dat"  # ROHSA Gaussian solution.
    FITS_FILE = "./data/processed/CRAFTS_cutout_baseline_K.fits"  # Baseline-subtracted input cube.
    OUTPUT_DIR = "./baseline/SNR=2/output_file_2sigma"  # Destination directory.
    
    # Optional line-free channel intervals used for noise estimation.
    NOISE_CHANNELS = [0, 190, 880, 990]  # Adjust to line-free channels in the local cube.
    
    # Minimum per-pixel amplitude signal-to-noise ratio.
    SNR_THRESHOLD = 2
    
    # Execute the configured workflow.
    filtered_results, masks = main(
        gaussian_file=GAUSSIAN_FILE,
        fits_file=FITS_FILE,
        output_dir=OUTPUT_DIR,
        noise_channels=NOISE_CHANNELS,
        snr_threshold=SNR_THRESHOLD
    )


## 5. Identify connected sources within each component

Read the filtered ROHSA parameters, identify spatially connected nonzero regions, reject regions smaller than `min_pixels`, and write masks, source tables, filtered products, and component-level diagnostic figures.


In [ ]:
# Purpose: Define the connected-component source-identification workflow.

# 1. Read the filtered DAT product in native and physical units.
def read_filtered_dat(dat_file):
    """Read an S/N-filtered ROHSA DAT file.

        Parameters are retained in both native pixel units and FITS-derived physical
        units, with component arrays stored in ``(x, y)`` order.

        Parameters
        ----------
        dat_file : str or path-like
            Filtered ROHSA DAT file.

        Returns
        -------
        dict
            Pixel- and physical-unit component dictionaries, component count,
            spatial shape, and original interleaved arrays."""
    print("\n" + "="*60)
    print("READING FILTERED ROHSA DAT FILE")
    print("="*60)
    
    # Read the native ROHSA parameter cube in pixel units.
    print("Reading Gaussian parameters (pixel units)...")
    gaussian_pixel = core.read_gaussian(dat_file)
    
    print(f"Gaussian pixel array shape: {gaussian_pixel.shape}")
    print(f"Data type: {gaussian_pixel.dtype}")
    
    # Convert channel coordinates and widths to physical velocity units.
    print("Converting to physical units...")
    gaussian_physical = core.physical_gaussian(gaussian_pixel)
    
    # Parse the interleaved array shape: (3 * n_components, n_y, n_x).
    n_params_times_comp, n_y, n_x = gaussian_pixel.shape
    n_components = n_params_times_comp // 3
    
    print(f"Spatial dimensions: {n_x} × {n_y}")
    print(f"Number of Gaussian components: {n_components}")
    
    # Extract each component in native pixel units.
    results_pixel = {}
    # Extract each component in physical units.
    results_physical = {}
    
    for comp in range(n_components):
        # Native ROHSA pixel units.
        amp_pixel = gaussian_pixel[3*comp]
        pos_pixel = gaussian_pixel[3*comp + 1]
        dis_pixel = gaussian_pixel[3*comp + 2]
        
        results_pixel[f'comp{comp+1}'] = {
            'amplitude': amp_pixel,
            'position': pos_pixel,
            'dispersion': dis_pixel
        }
        
        # Physical units used for scientific interpretation.
        amp_phys = gaussian_physical[3*comp]
        pos_phys = gaussian_physical[3*comp + 1]
        dis_phys = gaussian_physical[3*comp + 2]
        
        results_physical[f'comp{comp+1}'] = {
            'amplitude': amp_phys,
            'position': pos_phys,
            'dispersion': dis_phys
        }
        
        print(f"\nComponent {comp+1}:")
        print(f"  Pixel units:")
        print(f"    Amplitude range: [{np.min(amp_pixel):.4f}, {np.max(amp_pixel):.4f}]")
        print(f"    Position range: [{np.min(pos_pixel):.4f}, {np.max(pos_pixel):.4f}]")
        print(f"    Dispersion range: [{np.min(dis_pixel):.4f}, {np.max(dis_pixel):.4f}]")
        print(f"  Physical units:")
        print(f"    Amplitude range: [{np.min(amp_phys):.4f}, {np.max(amp_phys):.4f}] K")
        print(f"    Position range: [{np.min(pos_phys):.4f}, {np.max(pos_phys):.4f}] km/s")
        print(f"    Dispersion range: [{np.min(dis_phys):.4f}, {np.max(dis_phys):.4f}] km/s")
        print(f"    Non-zero pixels: {np.sum(amp_phys > 0)}")
    
    results = {
        'pixel': results_pixel,
        'physical': results_physical,
        'n_components': n_components,
        'shape': (n_x, n_y),
        'original_data_pixel': gaussian_pixel,
        'original_data_physical': gaussian_physical
    }
    
    return results

# 2. Identify connected sources within one Gaussian component.
def identify_sources_in_component(amplitude_map, min_pixels=10, max_gap=2):
    """Label spatially connected sources in one amplitude map.

        A binary dilation allows gaps up to ``max_gap`` pixels before connected-region
        labeling. Regions smaller than ``min_pixels`` are discarded and surviving
        labels are renumbered consecutively.

        Parameters
        ----------
        amplitude_map : numpy.ndarray
            Two-dimensional physical amplitude map in kelvin.
        min_pixels : int, default=10
            Minimum number of original nonzero pixels required for a source.
        max_gap : int, default=2
            Dilation scale used to bridge nearby detections.

        Returns
        -------
        labeled_mask : numpy.ndarray
            Integer source labels, with zero denoting background.
        n_sources : int
            Number of retained sources.
        source_properties : list of dict
            Pixel count, bounding box, centroid, peak, and integrated amplitude for
            each retained source."""
    print(f"\nIdentifying sources with min_pixels={min_pixels}, max_gap={max_gap}")
    
    # Define detections as pixels with positive filtered amplitude.
    binary_mask = amplitude_map > 0
    
    # Dilate detections to bridge gaps up to the configured scale.
    structure = ndimage.generate_binary_structure(2, 1)
    if max_gap > 1:
        structure = ndimage.iterate_structure(structure, max_gap - 1)
    
    # Label connected regions in the dilated mask.
    labeled_mask, num_features = ndimage.label(binary_mask, structure=structure)
    
    print(f"Initial connected components: {num_features}")
    
    # Reject regions with too few original detected pixels.
    source_properties = []
    final_mask = np.zeros_like(labeled_mask)
    source_count = 0
    
    for label in range(1, num_features + 1):
        mask = labeled_mask == label
        pixel_count = np.sum(mask)
        
        if pixel_count >= min_pixels:
            source_count += 1
            final_mask[mask] = source_count
            
            # Measure source geometry and amplitude statistics.
            y_indices, x_indices = np.where(mask)
            amp_values = amplitude_map[mask]
            
            props = {
                'source_id': source_count,
                'pixel_count': pixel_count,
                'centroid': (np.mean(x_indices), np.mean(y_indices)),
                'peak_amplitude': np.max(amp_values),  # K
                'mean_amplitude': np.mean(amp_values), # K
                'bbox': (np.min(x_indices), np.max(x_indices), 
                        np.min(y_indices), np.max(y_indices)),
                'pixels': list(zip(x_indices, y_indices))
            }
            source_properties.append(props)
            
            print(f"  Source {source_count}: {pixel_count} pixels, "
                  f"peak={props['peak_amplitude']:.2f} K, "
                  f"centroid=({props['centroid'][0]:.1f}, {props['centroid'][1]:.1f})")
    
    print(f"Final sources after filtering: {source_count}")
    
    return final_mask, source_count, source_properties

# 3. Plot source masks and physical parameter maps by component.
def plot_component_sources(component_data_phys, component_idx, output_dir, min_pixels=10, max_gap=2):
    """Identify and visualize sources in one Gaussian component.

        Parameters
        ----------
        component_data_phys : dict
            Physical amplitude, centroid, and dispersion maps for one component.
        component_idx : int
            Zero-based component index.
        output_dir : str or path-like
            Directory for the diagnostic PNG.
        min_pixels, max_gap : int
            Connected-source selection parameters.

        Returns
        -------
        source_mask : numpy.ndarray
            Integer source-label mask.
        n_sources : int
            Number of retained sources.
        source_props : list of dict
            Measured properties for each source."""
    amplitude = component_data_phys['amplitude']      # K
    position = component_data_phys['position']        # km/s
    dispersion = component_data_phys['dispersion']    # km/s
    
    print(f"\nPlotting component {component_idx} with physical units:")
    print(f"  Amplitude range: [{np.min(amplitude[amplitude>0]):.4f}, {np.max(amplitude):.4f}] K")
    print(f"  Position range: [{np.min(position[position!=0]):.4f}, {np.max(position):.4f}] km/s")
    print(f"  Dispersion range: [{np.min(dispersion[dispersion>0]):.4f}, {np.max(dispersion):.4f}] km/s")
    
    # Identify connected sources from the component amplitude map.
    source_mask, n_sources, source_props = identify_sources_in_component(
        amplitude, min_pixels=min_pixels, max_gap=max_gap
    )
    
    # Derive valid color limits for each physical parameter.
    amp_vmin = np.min(amplitude[amplitude>0]) if np.any(amplitude>0) else 0
    amp_vmax = np.max(amplitude)
    pos_vmin = np.min(position[position!=0]) if np.any(position!=0) else -10
    pos_vmax = np.max(position)
    dis_vmin = np.min(dispersion[dispersion>0]) if np.any(dispersion>0) else 0
    dis_vmax = np.max(dispersion)
    
    # Arrange original and filtered maps in a 2-by-3 grid.
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Top row: unfiltered component parameters.
    # Original amplitude.
    im1 = axes[0, 0].imshow(amplitude, origin='lower', cmap='inferno', 
                            vmin=amp_vmin, vmax=amp_vmax)
    axes[0, 0].set_title(f'Component {component_idx} - Original Amplitude', fontsize=12)
    cbar1 = plt.colorbar(im1, ax=axes[0, 0], shrink=0.8, pad=0.05, location='bottom')
    cbar1.set_label('Brightness Temp. [K]', fontsize=10)
    
    # Integer source-label mask.
    from matplotlib.colors import ListedColormap
    ax2 = axes[0, 1]
    if n_sources > 0:
        colors = plt.cm.tab20(np.linspace(0, 1, n_sources))
        colors = np.vstack([[0,0,0,1], colors])
        cmap = ListedColormap(colors)
        
        im2 = ax2.imshow(source_mask, origin='lower', cmap=cmap, 
                         vmin=0, vmax=n_sources, interpolation='nearest')
        ax2.set_title(f'Component {component_idx} - Source Mask ({n_sources} sources)', fontsize=12)
        cbar2 = plt.colorbar(im2, ax=ax2, shrink=0.8, pad=0.05, location='bottom', ticks=range(n_sources+1))
        cbar2.set_label('Source ID', fontsize=10)
        
        # Annotate each retained source at its measured centroid.
        for props in source_props:
            x, y = props['centroid']
            ax2.plot(x, y, 'r+', markersize=10, markeredgewidth=2)
            ax2.text(x+2, y+2, f'{props["source_id"]}', 
                    color='white', fontsize=10, weight='bold',
                    bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))
    else:
        ax2.imshow(source_mask, origin='lower', cmap='gray')
        ax2.set_title(f'Component {component_idx} - No Sources Found', fontsize=12)
    
    # Original centroid-velocity map.
    im3 = axes[0, 2].imshow(position, origin='lower', cmap='coolwarm',
                            vmin=pos_vmin, vmax=pos_vmax)
    axes[0, 2].set_title(f'Component {component_idx} - Original Position', fontsize=12)
    cbar3 = plt.colorbar(im3, ax=axes[0, 2], shrink=0.8, pad=0.05, location='bottom')
    cbar3.set_label('Velocity [km s$^{-1}$]', fontsize=10)
    
    # Bottom row: physical parameters retained inside source masks.
    # Source-masked amplitude map.
    filtered_amplitude = np.zeros_like(amplitude)
    filtered_amplitude[source_mask > 0] = amplitude[source_mask > 0]
    
    im4 = axes[1, 0].imshow(filtered_amplitude, origin='lower', cmap='inferno',
                            vmin=amp_vmin, vmax=amp_vmax)
    axes[1, 0].set_title(f'Component {component_idx} - Source-Only Amplitude', fontsize=12)
    cbar4 = plt.colorbar(im4, ax=axes[1, 0], shrink=0.8, pad=0.05, location='bottom')
    cbar4.set_label('Brightness Temp. [K]', fontsize=10)
    
    # Source-masked centroid-velocity map.
    filtered_position = np.zeros_like(position)
    filtered_position[source_mask > 0] = position[source_mask > 0]
    
    im5 = axes[1, 1].imshow(filtered_position, origin='lower', cmap='coolwarm',
                            vmin=pos_vmin, vmax=pos_vmax)
    axes[1, 1].set_title(f'Component {component_idx} - Source-Only Position', fontsize=12)
    cbar5 = plt.colorbar(im5, ax=axes[1, 1], shrink=0.8, pad=0.05, location='bottom')
    cbar5.set_label('Velocity [km s$^{-1}$]', fontsize=10)
    
    # Source-masked velocity-dispersion map.
    filtered_dispersion = np.zeros_like(dispersion)
    filtered_dispersion[source_mask > 0] = dispersion[source_mask > 0]
    
    im6 = axes[1, 2].imshow(filtered_dispersion, origin='lower', cmap='cubehelix',
                            vmin=dis_vmin, vmax=dis_vmax)
    axes[1, 2].set_title(f'Component {component_idx} - Source-Only Dispersion', fontsize=12)
    cbar6 = plt.colorbar(im6, ax=axes[1, 2], shrink=0.8, pad=0.05, location='bottom')
    cbar6.set_label('Dispersion [km s$^{-1}$]', fontsize=10)
    
    # Record the component index and applied S/N threshold.
    plt.suptitle(f'Component {component_idx} Source Identification\n'
                f'Min pixels: {min_pixels}, Max gap: {max_gap}\n'
                f'Amplitude: K | Position: km/s | Dispersion: km/s', 
                fontsize=14, y=1.02)
    
    plt.tight_layout()
    
    # Save and close the figure to release memory during batch runs.
    comp_file = os.path.join(output_dir, f'component{component_idx}_sources.png')
    plt.savefig(comp_file, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"\nComponent {component_idx} source plot saved to: {comp_file}")
    
    return source_mask, n_sources, source_props

# 4. Save source-masked physical parameters to FITS.
def save_source_filtered_fits_physical(results, all_source_masks, output_file):
    """Save source-masked parameters as a physical-unit FITS cube.

        Parameters
        ----------
        results : dict
            Filtered ROHSA result structure.
        all_source_masks : sequence of numpy.ndarray
            Integer source masks for all Gaussian components.
        output_file : str or path-like
            Destination FITS path.

        Returns
        -------
        bool
            ``True`` on success and ``False`` on failure."""
    try:
        n_components = results['n_components']
        n_y, n_x = results['shape'][1], results['shape'][0]
        
        # Assemble interleaved physical parameter planes for all components.
        filtered_data_phys = np.zeros((3*n_components, n_y, n_x))
        
        for comp in range(n_components):
            source_mask = all_source_masks[comp]
            comp_data = results['physical'][f'comp{comp+1}']
            
            filtered_data_phys[3*comp] = comp_data['amplitude'] * (source_mask > 0)
            filtered_data_phys[3*comp + 1] = comp_data['position'] * (source_mask > 0)
            filtered_data_phys[3*comp + 2] = comp_data['dispersion'] * (source_mask > 0)
        
        hdu = fits.PrimaryHDU(filtered_data_phys)
        hdu.writeto(output_file, overwrite=True)
        print(f"\nSource-filtered FITS file (physical units) saved to: {output_file}")
        return True
    except Exception as e:
        print(f"Error saving source-filtered FITS: {e}")
        return False

# 5. Save source-masked native parameters to DAT.
def save_source_filtered_dat_pixel(results, all_source_masks, original_dat_file, output_file):
    """Save source-masked parameters in native ROHSA DAT format.

        Parameters
        ----------
        results : dict
            Filtered ROHSA result structure.
        all_source_masks : sequence of numpy.ndarray
            Integer source masks for all components.
        original_dat_file : str or path-like
            DAT file used to preserve the original header.
        output_file : str or path-like
            Destination DAT path.

        Returns
        -------
        bool
            ``True`` on success and ``False`` on failure."""
    print("\n" + "="*60)
    print("SAVING SOURCE-FILTERED RESULTS TO .DAT FORMAT (PIXEL UNITS)")
    print("="*60)
    
    try:
        n_components = results['n_components']
        n_y, n_x = results['shape'][1], results['shape'][0]
        
        print(f"Reorganizing data for .dat format...")
        print(f"  Spatial grid: X({n_x}) × Y({n_y})")
        print(f"  Components: {n_components}")
        print(f"  DAT file format: Y X AMP POS DISP (Y first, X second)")
        print(f"  Units: Pixel units (matching original ROHSA format)")
        
        # Preserve header lines from the original ROHSA file.
        with open(original_dat_file, 'r') as f:
            header_lines = []
            for i in range(27):
                line = f.readline()
                header_lines.append(line)
        
        # Apply source masks to copies of the native parameter arrays.
        filtered_data_pixel = np.zeros((3*n_components, n_y, n_x))
        
        for comp in range(n_components):
            source_mask = all_source_masks[comp]
            comp_data_pixel = results['pixel'][f'comp{comp+1}']
            
            filtered_data_pixel[3*comp] = comp_data_pixel['amplitude'] * (source_mask > 0)
            filtered_data_pixel[3*comp + 1] = comp_data_pixel['position'] * (source_mask > 0)
            filtered_data_pixel[3*comp + 2] = comp_data_pixel['dispersion'] * (source_mask > 0)
        
        # Count retained component entries.
        total_entries = n_x * n_y * n_components
        nonzero_entries = np.sum(filtered_data_pixel != 0)
        print(f"  Non-zero entries: {nonzero_entries} ({100 * nonzero_entries / total_entries:.2f}%)")
        
        # Generate rows in native ROHSA order: y, x, then parameters.
        data_lines = []
        for j in range(n_y):  # Outer loop over y.
            for i in range(n_x):  # Inner loop over x.
                for comp in range(n_components):
                    amp = filtered_data_pixel[3*comp, j, i]
                    mean = filtered_data_pixel[3*comp + 1, j, i]
                    sigma = filtered_data_pixel[3*comp + 2, j, i]
                    
                    line = f"    {j:4d}    {i:4d}    {amp:20.16f}    {mean:20.16f}    {sigma:20.16f}\n"
                    data_lines.append(line)
        
        # Write the header and all parameter rows.
        with open(output_file, 'w') as f:
            f.writelines(header_lines)
            f.writelines(data_lines)
        
        print(f"\n.dat file statistics:")
        print(f"  Total entries: {total_entries}")
        print(f"  Non-zero entries: {nonzero_entries}")
        print(f"  Data density: {100 * nonzero_entries / total_entries:.2f}%")
        print(f"\nSource-filtered .dat file (pixel units) saved to: {output_file}")
        
        # Preview a small number of data rows as a write check.
        print(f"\n  First few lines of saved file:")
        with open(output_file, 'r') as f:
            for i, line in enumerate(f):
                if i >= 5:  # Preview five data rows.
                    break
                if i >= len(header_lines):  # Skip header lines.
                    print(f"    {line.strip()}")
        
        return True
        
    except Exception as e:
        print(f"Error saving source-filtered .dat: {e}")
        import traceback
        traceback.print_exc()
        return False

# 6. Plot a summary of source labels across all components.
def plot_summary(all_source_masks, all_source_counts, output_dir):
    """Plot source-label maps for all Gaussian components.

        Parameters
        ----------
        all_source_masks : sequence of numpy.ndarray
            Integer source masks in component order.
        all_source_counts : sequence of int
            Number of retained sources in each component.
        output_dir : str or path-like
            Directory for the summary PNG."""
    n_components = len(all_source_masks)
    
    fig, axes = plt.subplots(1, n_components, figsize=(5*n_components, 5))
    if n_components == 1:
        axes = [axes]
    
    from matplotlib.colors import ListedColormap
    
    for comp in range(n_components):
        source_mask = all_source_masks[comp]
        n_sources = all_source_counts[comp]
        
        if n_sources > 0:
            colors = plt.cm.tab20(np.linspace(0, 1, n_sources))
            colors = np.vstack([[0,0,0,1], colors])
            cmap = ListedColormap(colors)
            
            im = axes[comp].imshow(source_mask, origin='lower', cmap=cmap,
                                   vmin=0, vmax=n_sources, aspect='auto', interpolation='nearest')
            axes[comp].set_title(f'Component {comp+1}\n{n_sources} sources', fontsize=12)
            cbar = plt.colorbar(im, ax=axes[comp], shrink=0.8, pad=0.15, location='bottom')
            cbar.set_label('Source ID', fontsize=10)
            cbar.set_ticks(range(n_sources+1))
        else:
            im = axes[comp].imshow(source_mask, origin='lower', cmap='gray', aspect='auto')
            axes[comp].set_title(f'Component {comp+1}\nNo sources', fontsize=12)
        
        axes[comp].set_xlabel('X pixel')
        axes[comp].set_ylabel('Y pixel')
    
    plt.suptitle('Source Identification Results - All Components', fontsize=14, y=1.02)
    plt.tight_layout()
    
    summary_file = os.path.join(output_dir, 'all_components_sources_summary.png')
    plt.savefig(summary_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\nSummary plot saved to: {summary_file}")

# 7. Save measured source properties to a text report.
def save_source_info(all_source_props, output_dir):
    """Write per-component source properties to a text file.

        Parameters
        ----------
        all_source_props : sequence of list of dict
            Source-property records grouped by Gaussian component.
        output_dir : str or path-like
            Directory for the output report."""
    info_file = os.path.join(output_dir, 'source_information.txt')
    
    with open(info_file, 'w') as f:
        f.write("="*60 + "\n")
        f.write("SOURCE IDENTIFICATION RESULTS\n")
        f.write("="*60 + "\n\n")
        f.write("All values are in physical units:\n")
        f.write("  - Amplitude: K (Kelvin)\n")
        f.write("  - Position: km/s\n")
        f.write("  - Dispersion: km/s\n\n")
        
        for comp_idx, source_props in enumerate(all_source_props):
            f.write(f"\n{'='*40}\n")
            f.write(f"COMPONENT {comp_idx+1}\n")
            f.write(f"{'='*40}\n\n")
            
            if len(source_props) == 0:
                f.write("No sources found in this component.\n")
                continue
            
            for props in source_props:
                f.write(f"Source {props['source_id']}:\n")
                f.write(f"  Number of pixels: {props['pixel_count']}\n")
                f.write(f"  Centroid (X, Y): ({props['centroid'][0]:.2f}, {props['centroid'][1]:.2f})\n")
                f.write(f"  Peak amplitude: {props['peak_amplitude']:.4f} K\n")
                f.write(f"  Mean amplitude: {props['mean_amplitude']:.4f} K\n")
                f.write(f"  Bounding box: X[{props['bbox'][0]}, {props['bbox'][1]}], "
                       f"Y[{props['bbox'][2]}, {props['bbox'][3]}]\n\n")
    
    print(f"Source information saved to: {info_file}")

# 8. Save all integer source masks to FITS.
def save_source_masks(all_source_masks, output_file):
    """Write all component source-label masks to a FITS cube.

        Parameters
        ----------
        all_source_masks : sequence of numpy.ndarray
            Integer source masks in component order.
        output_file : str or path-like
            Destination FITS path."""
    masks_array = np.array(all_source_masks)
    hdu = fits.PrimaryHDU(masks_array)
    hdu.writeto(output_file, overwrite=True)
    print(f"Source masks saved to: {output_file}")

# 9. Orchestrate connected-component source identification.
def main(filtered_dat_file, original_dat_file, output_dir, 
         min_pixels=10, max_gap=2):
    """Run connected-component source identification.

        Parameters
        ----------
        filtered_dat_file : str or path-like
            S/N-filtered ROHSA DAT product.
        original_dat_file : str or path-like
            Original DAT file used to preserve header metadata.
        output_dir : str or path-like
            Directory for masks, tables, filtered products, and figures.
        min_pixels : int, default=10
            Minimum number of detected pixels required for a source.
        max_gap : int, default=2
            Dilation scale used to bridge nearby detections.

        Returns
        -------
        all_source_masks : list of numpy.ndarray
            Integer source-label masks in component order.
        all_source_props : list of list of dict
            Source-property records grouped by Gaussian component."""
    
    print("\n" + "="*60)
    print("SOURCE IDENTIFICATION FROM FILTERED ROHSA RESULTS")
    print("="*60)
    print(f"Filtered DAT file: {filtered_dat_file}")
    print(f"Output directory: {output_dir}")
    print(f"Min pixels per source: {min_pixels}")
    print(f"Max gap between pixels: {max_gap}")
    
    # Create output directories before any file-producing operation.
    os.makedirs(output_dir, exist_ok=True)
    
    # Read filtered Gaussian parameters in both unit systems.
    results = read_filtered_dat(filtered_dat_file)
    n_components = results['n_components']
    
    # Accumulate masks, counts, and properties in component order.
    all_source_masks = []
    all_source_counts = []
    all_source_props = []
    
    # Identify and plot sources using physical parameter maps.
    for comp in range(n_components):
        print(f"\n{'='*40}")
        print(f"PROCESSING COMPONENT {comp+1}")
        print(f"{'='*40}")
        
        component_data_phys = results['physical'][f'comp{comp+1}']
        
        # Run source labeling and save the component diagnostic.
        source_mask, n_sources, source_props = plot_component_sources(
            component_data_phys, comp+1, output_dir, 
            min_pixels=min_pixels, max_gap=max_gap
        )
        
        all_source_masks.append(source_mask)
        all_source_counts.append(n_sources)
        all_source_props.append(source_props)
    
    # Write the source-masked physical FITS product.
    source_filtered_fits_phys = os.path.join(output_dir, 'source_filtered_rohsa_physical.fits')
    save_source_filtered_fits_physical(results, all_source_masks, source_filtered_fits_phys)
    
    # Write integer source labels for all components.
    save_source_masks(all_source_masks, os.path.join(output_dir, 'source_masks.fits'))
    
    # Write the source-masked native DAT product.
    source_filtered_dat_pixel = os.path.join(output_dir, 'source_filtered_rohsa_pixel.dat')
    save_source_filtered_dat_pixel(results, all_source_masks, original_dat_file, 
                                  source_filtered_dat_pixel)
    
    # Save the all-component source-label summary.
    plot_summary(all_source_masks, all_source_counts, output_dir)
    
    # Save the source-property report.
    save_source_info(all_source_props, output_dir)
    
    # Report output products and source counts.
    print("\n" + "="*60)
    print("SOURCE IDENTIFICATION COMPLETED")
    print("="*60)
    print(f"\nSummary:")
    for comp in range(n_components):
        print(f"  Component {comp+1}: {all_source_counts[comp]} sources identified")
    
    print(f"\nOutput files saved to: {output_dir}")
    print(f"  - Source-filtered FITS (physical): source_filtered_rohsa_physical.fits")
    print(f"  - Source-filtered DAT (pixel): source_filtered_rohsa_pixel.dat")
    print(f"  - Source masks: source_masks.fits")
    print(f"  - Individual component plots: component*_sources.png")
    print(f"  - Summary plot: all_components_sources_summary.png")
    print(f"  - Source information: source_information.txt")
    
    return all_source_masks, all_source_props

# 10. Configure and run source identification.
if __name__ == "__main__":
    # Update these paths for the local data layout.
    FILTERED_DAT_FILE = "./baseline/SNR=2/output_file_2sigma/filtered_rohsa.dat"  # DAT product from the S/N-filtering stage.
    ORIGINAL_DAT_FILE = "./baseline/ROHSA_3ngauss_3D_1.1.1.0.dat"  # Original DAT file used as the header template.
    OUTPUT_DIR = "./baseline/SNR=2/output_individual_source_10"  # Destination directory.
    
    # Connected-source selection parameters.
    MIN_PIXELS = 10  # Minimum detected pixels per source.
    MAX_GAP = 2      # Maximum gap bridged within a source.
    
    # Execute the configured workflow.
    all_source_masks, all_source_props = main(
        filtered_dat_file=FILTERED_DAT_FILE,
        original_dat_file=ORIGINAL_DAT_FILE,
        output_dir=OUTPUT_DIR,
        min_pixels=MIN_PIXELS,
        max_gap=MAX_GAP
    )


## 6. Merge component-level sources

Enable automatic module reloading and import the external merge utilities. This keeps notebook runs synchronized with edits to `merge_utils.py`.


In [ ]:
# Purpose: Load the source-merging utilities with IPython autoreload enabled.

# ============================================================
# Reload local Python modules automatically during interactive development.
# ============================================================
%load_ext autoreload
%autoreload 2

import os
# import merge_utils as mu
import merge_utils as mu
from astropy.io import fits

from ROHSApy import ROHSA


### Configure source merging

Set input paths, output location, connected-source criteria, spatial-overlap threshold, and velocity-consistency threshold, then verify that required inputs exist.


In [ ]:
# Purpose: Declare and validate source-merging inputs and parameters.

# ============================================================
# Configure merge inputs, outputs, and acceptance thresholds.
# ============================================================
DAT_FILE = "./baseline/SNR=2/output_individual_source_10/source_filtered_rohsa_pixel.dat"
ORIGINAL_DAT_FILE = "./baseline/ROHSA_3ngauss_3D_1.1.1.0.dat"

# FITS cube supplying the celestial WCS used in output products.
FITS_FILE = fitsname

OUTPUT_DIR = "./baseline/SNR=2/output_individual_source_10/output_merged_source_0.7"

MIN_PIXELS = 10
MAX_GAP = 2

MIN_OVERLAP_PIXELS = 2
VELOCITY_THRESHOLD_FACTOR =0.7

HEADER_NLINES = 27

print("DAT exists:", os.path.exists(DAT_FILE))
print("Original DAT exists:", os.path.exists(ORIGINAL_DAT_FILE))
print("FITS exists:", os.path.exists(FITS_FILE))
print("Output dir:", OUTPUT_DIR)


### Run source merging

Merge component-level detections into final HVC sources and generate the associated tables, masks, and diagnostic figures.


In [ ]:
# Purpose: Execute the configured source-merging workflow.

# ============================================================
# Run automated source merging and diagnostic plotting.
# ============================================================
mu.main(
    core=core,
    dat_file=DAT_FILE,
    original_dat_file=ORIGINAL_DAT_FILE,
    output_dir=OUTPUT_DIR,
    fits_file=FITS_FILE,
    min_pixels=MIN_PIXELS,
    max_gap=MAX_GAP,
    min_overlap_pixels=MIN_OVERLAP_PIXELS,
    velocity_threshold_factor=VELOCITY_THRESHOLD_FACTOR,
    header_nlines=HEADER_NLINES,
)


## 7. Calculate physical source parameters

Run the repository script that derives integrated H I observables and uncertainties for the merged sources. Review the path and physical assumptions in that script before reuse.


In [ ]:
# Purpose: Execute the physical-parameter calculation script.

%run calculate_hvc_physical_parameters_v2.py


## 8. Generate per-source summary figures

Create the standard 2-by-2 diagnostic summaries for each final source, including moment maps and line-width information.


In [ ]:
# Purpose: Execute the per-source summary-figure script.

%run source_summary_2x2_auto_aspect_v2.py


## 9. Generate position-velocity diagrams

Run the PV extraction and plotting workflow for the source groups and paths configured inside `pv.py`.


In [ ]:
# Purpose: Execute the position-velocity analysis script.

%run pv.py


## 10. Inspect individual spectra

Load the CRAFTS cube as a `SpectralCube`, preserving its WCS and units for pixel-level spectrum extraction and Gaussian-fit diagnostics.


In [ ]:
# Purpose: Load the H I cube as a unit-aware SpectralCube.

import os
import glob
import subprocess
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from astropy.io import fits
from spectral_cube import SpectralCube
import astropy.units as u
from astropy.modeling.models import Gaussian1D
from matplotlib.ticker import FormatStrFormatter
from matplotlib.lines import Line2D

cube = SpectralCube.read('./data/processed/CRAFTS_cutout_baseline_K.fits')  # Initiate a SpectralCube
#hi_data.close()  # Close the FITS file - we already read it in and don't need it anymore!
print(cube)


### Preview a selected spectrum

Convert the spectral axis to km/s and display a quick-look spectrum at the selected `(y, x)` pixel coordinate.


In [ ]:
# Purpose: Extract and preview one spectrum in velocity units.

cube = cube.with_spectral_unit(u.km / u.s)
y, x = 11,80

spec = cube[:, y, x]
# plt.xlim(-350000,-150000)
spec.quicklook()


### Plot Gaussian components, residuals, and reduced chi-square

Load per-pixel Gaussian parameters from the merged-source CSV files, reconstruct the fit for a selected pixel, estimate line-free noise, compute reduced chi-square, and save a publication-oriented spectrum and residual figure.


In [ ]:
# Purpose: Define and run the merged-source spectral-fit diagnostic.

import os
import glob
import subprocess
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from astropy.io import fits
from spectral_cube import SpectralCube
import astropy.units as u
from astropy.modeling.models import Gaussian1D
from matplotlib.ticker import FormatStrFormatter
from matplotlib.lines import Line2D


# =========================================================
# 0. Configure publication-oriented typography.
# =========================================================
def set_times_new_roman_font():
    """Configure a publication-oriented serif font for Matplotlib.

        The function searches common platform-specific paths, then queries ``fc-match``.
        If Times New Roman is unavailable, it falls back to a portable serif stack and
        configures embedded TrueType fonts for PDF and PostScript output."""
    font_loaded = False
    loaded_font_name = None

    possible_paths = [
        r"C:\Windows\Fonts\times.ttf",
        r"C:\Windows\Fonts\timesbd.ttf",
        r"C:\Windows\Fonts\timesi.ttf",
        r"C:\Windows\Fonts\timesbi.ttf",
        "/usr/share/fonts/truetype/msttcorefonts/Times_New_Roman.ttf",
        "/usr/share/fonts/truetype/msttcorefonts/times.ttf",
        "/usr/share/fonts/TTF/Times_New_Roman.ttf",
        "/usr/local/share/fonts/Times_New_Roman.ttf",
    ]

    for path in possible_paths:
        if os.path.exists(path):
            fm.fontManager.addfont(path)
            loaded_font_name = fm.FontProperties(fname=path).get_name()
            plt.rcParams["font.family"] = loaded_font_name
            mpl.rcParams["font.family"] = loaded_font_name
            font_loaded = True
            print(f"Loaded font file: {path}")
            print(f"Matplotlib font name: {loaded_font_name}")
            break

    if not font_loaded:
        try:
            font_path = subprocess.check_output(
                ["fc-match", "-f", "%{file}", "Times New Roman"],
                text=True
            ).strip()

            if font_path and os.path.exists(font_path):
                fm.fontManager.addfont(font_path)
                loaded_font_name = fm.FontProperties(fname=font_path).get_name()
                plt.rcParams["font.family"] = loaded_font_name
                mpl.rcParams["font.family"] = loaded_font_name
                font_loaded = True
                print(f"fc-match returned font file: {font_path}")
                print(f"Matplotlib font name: {loaded_font_name}")

        except Exception as e:
            print(f"fc-match failed or Times New Roman not found: {e}")

    if not font_loaded:
        print("Times New Roman not found. Falling back to serif fonts.")
        plt.rcParams["font.family"] = "serif"
        mpl.rcParams["font.family"] = "serif"
        plt.rcParams["font.serif"] = [
            "Times New Roman",
            "Liberation Serif",
            "DejaVu Serif",
            "Times",
        ]
        mpl.rcParams["font.serif"] = [
            "Times New Roman",
            "Liberation Serif",
            "DejaVu Serif",
            "Times",
        ]

    plt.rcParams["mathtext.fontset"] = "stix"
    mpl.rcParams["mathtext.fontset"] = "stix"
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42
    mpl.rcParams["pdf.fonttype"] = 42
    mpl.rcParams["ps.fonttype"] = 42
    plt.rcParams["axes.unicode_minus"] = False
    mpl.rcParams["axes.unicode_minus"] = False


set_times_new_roman_font()


# =========================================================
# 1. Load the H I spectral cube.
# =========================================================
cube_file = "./data/processed/CRAFTS_cutout_baseline_K.fits"

cube = SpectralCube.read(cube_file)
cube = cube.with_spectral_unit(u.km / u.s)

print(cube)


# =========================================================
# 2. Load fitted parameters from merged-source CSV products.
# =========================================================
def standardize_columns(df):
    """Normalize merged-source CSV column names.

        Parameters
        ----------
        df : pandas.DataFrame
            Table whose columns may differ in case or contain a known legacy spelling.

        Returns
        -------
        pandas.DataFrame
            Copy with recognized fields renamed to the canonical schema."""

    rename_dict = {}

    for col in df.columns:
        low = col.lower().strip()

        if low == "final_source_id":
            rename_dict[col] = "final_source_id"
        elif low == "component":
            rename_dict[col] = "component"
        elif low in [
            "original_component_source_id",
            "original_componnet_source_id",
            "component_source_id",
        ]:
            rename_dict[col] = "original_component_source_id"
        elif low == "x_pixel":
            rename_dict[col] = "x_pixel"
        elif low == "y_pixel":
            rename_dict[col] = "y_pixel"
        elif low == "amplitude_k":
            rename_dict[col] = "amplitude_K"
        elif low == "velocity_kms":
            rename_dict[col] = "velocity_kms"
        elif low == "sigma_kms":
            rename_dict[col] = "sigma_kms"
        elif low == "fwhm_kms":
            rename_dict[col] = "fwhm_kms"
        elif low == "moment0_k_kms":
            rename_dict[col] = "moment0_K_kms"

    return df.rename(columns=rename_dict)


def read_merged_source_csv_to_rohsas_format(
    source_csv_dir="./baseline/SNR=2/output_individual_source_10/output_merged_source_0.7/source_physical_parameters"
):
    """Load merged-source Gaussian parameters from per-source CSV files.

        Parameters
        ----------
        source_csv_dir : str or path-like
            Directory containing ``source_*_physical_parameters.csv`` files.

        Returns
        -------
        dict
            Mapping ``(y_pixel, x_pixel, component, original_source_id)`` to final
            source ID, unit-aware Gaussian parameters, and source CSV provenance.

        Raises
        ------
        FileNotFoundError
            If no matching source CSV files are found.
        ValueError
            If a required column is absent."""

    result_dict = {}

    csv_files = sorted(
        glob.glob(os.path.join(source_csv_dir, "source_*_physical_parameters.csv"))
    )

    if len(csv_files) == 0:
        raise FileNotFoundError(f"No source csv found in: {source_csv_dir}")

    print(f"Found {len(csv_files)} source CSV files.")

    for csv_file in csv_files:
        df = pd.read_csv(csv_file)
        df = standardize_columns(df)

        required_cols = [
            "final_source_id",
            "component",
            "original_component_source_id",
            "x_pixel",
            "y_pixel",
            "amplitude_K",
            "velocity_kms",
            "sigma_kms",
            "fwhm_kms",
        ]

        for col in required_cols:
            if col not in df.columns:
                raise ValueError(f"{csv_file} missing column: {col}")

        df = df.replace([np.inf, -np.inf], np.nan)
        df = df.dropna(subset=required_cols)
        df = df[df["amplitude_K"] > 0]

        for _, row in df.iterrows():
            source_id = int(row["final_source_id"])
            comp = int(row["component"])
            original_source_id = int(row["original_component_source_id"])

            x = int(row["x_pixel"])
            y = int(row["y_pixel"])

            # Preserve the notebook convention i=y and j=x in dictionary keys.
            i = y
            j = x

            key = (i, j, comp, original_source_id)

            result_dict[key] = {
                "source_id": source_id,
                "component": comp,
                "original_component_source_id": original_source_id,
                "amplitude": float(row["amplitude_K"]) * u.K,
                "mean": float(row["velocity_kms"]) * u.km / u.s,
                "stddev": float(row["sigma_kms"]) * u.km / u.s,
                "fwhm": float(row["fwhm_kms"]) * u.km / u.s,
                "csv_file": csv_file,
            }

    print(f"Loaded {len(result_dict)} fitted Gaussian components from merged-source CSVs.")

    return result_dict


# =========================================================
# 3. Query all fitted components at a selected pixel.
# =========================================================
def get_all_components_at_pixel(result_dict, i, j, source_id=None):
    """Select fitted Gaussian components at one image pixel.

        Parameters
        ----------
        result_dict : dict
            Mapping returned by :func:`read_merged_source_csv_to_rohsas_format`.
        i, j : int
            Pixel coordinates, where ``i`` is y and ``j`` is x.
        source_id : int, optional
            Restrict the result to one final merged source.

        Returns
        -------
        dict
            Mapping ``(component, original_component_source_id)`` to parameter records."""

    comps = {}

    for key, vals in result_dict.items():
        y, x, comp, orig_src = key

        if y == i and x == j:
            if source_id is not None and vals["source_id"] != source_id:
                continue

            comps[(comp, orig_src)] = vals

    return comps


def print_available_components_at_pixel(result_dict, i, j):
    """Tabulate fitted components available at one pixel.

        Parameters
        ----------
        result_dict : dict
            Mapping returned by :func:`read_merged_source_csv_to_rohsas_format`.
        i, j : int
            Pixel coordinates, where ``i`` is y and ``j`` is x.

        Returns
        -------
        pandas.DataFrame
            Sorted component table, or an empty table when no fit is available."""

    rows = []

    for key, vals in result_dict.items():
        y, x, comp, orig_src = key

        if y == i and x == j:
            rows.append(
                {
                    "final_source_id": vals["source_id"],
                    "component": comp,
                    "original_component_source_id": orig_src,
                    "Amplitude_K": vals["amplitude"].value,
                    "Velocity_kms": vals["mean"].value,
                    "Sigma_kms": vals["stddev"].value,
                    "FWHM_kms": vals["fwhm"].value,
                }
            )

    if len(rows) == 0:
        print(f"No fit data found at pixel (i={i}, j={j})")
        return pd.DataFrame()

    df = pd.DataFrame(rows).sort_values(
        ["final_source_id", "component", "original_component_source_id"]
    )

    print(f"Fit components at pixel (i={i}, j={j}):")
    print(df)

    return df


# =========================================================
# 4. Plot the fit, residual, and reduced chi-square diagnostic.
# =========================================================
def plot_with_residual_and_chi2(
    cube,
    result_dict,
    i,
    j,
    source_id=None,
    output_dir="./baseline/SNR=2/output_individual_source_10/output_merged_source_0.7/spectra_plots",
    xlim=(-350, -150),
    ylim=(-0.4, 1),
):
    """Plot a spectrum, Gaussian decomposition, residuals, and fit statistic.

        Noise is estimated from channels with velocities below -305 km/s or above
        -195 km/s. Reduced chi-square uses three free parameters per included Gaussian.

        Parameters
        ----------
        cube : spectral_cube.SpectralCube
            Unit-aware H I cube.
        result_dict : dict
            Per-pixel Gaussian records loaded from merged-source CSV files.
        i, j : int
            Pixel coordinates, where ``i`` is y and ``j`` is x.
        source_id : int, optional
            Restrict the model to one final source; include all sources when omitted.
        output_dir : str or path-like
            Directory for the output PDF.
        xlim, ylim : tuple of float
            Display limits for velocity and brightness temperature.

        Returns
        -------
        float or None
            Reduced chi-square, or ``None`` when no fit or valid noise estimate exists."""

    os.makedirs(output_dir, exist_ok=True)

    # Extract the observed spectrum and discard non-finite channels.
    spec = cube[:, i, j]
    x = cube.spectral_axis.to(u.km / u.s).value
    y = spec.value

    mask = np.isfinite(y)
    x = x[mask]
    y = y[mask]

    # Select Gaussian components associated with the requested pixel and source.
    comps = get_all_components_at_pixel(
        result_dict,
        i,
        j,
        source_id=source_id,
    )

    if not comps:
        print(f"No fit data for pixel (i={i}, j={j}), source_id={source_id}")
        return None

    # Evaluate each Gaussian and accumulate the total spectral model.
    total_model = np.zeros_like(x)
    models = []

    colors = ["cyan", "magenta", "green", "blue", "purple", "orange"]

    for idx, ((comp, orig_src), vals) in enumerate(sorted(comps.items())):
        g = Gaussian1D(
            amplitude=vals["amplitude"].value,
            mean=vals["mean"].value,
            stddev=vals["stddev"].value,
        )
    
        y_model = g(x)
        total_model += y_model
    
        label = f"Comp {idx+1}"


        models.append(
            {
                "component": comp,
                "original_component_source_id": orig_src,
                "source_id": vals["source_id"],
                "model": y_model,
                "color": colors[idx % len(colors)],
                "label": label,
            }
        )

    # Estimate RMS noise from velocity intervals outside the target emission.
    # The upper line-free interval begins at -195 km/s, not +195 km/s.
    noise_mask = (x < -305) | (x > -195)

    if np.sum(noise_mask) < 5:
        print("Not enough channels for noise estimation.")
        sigma = np.nanstd(y)
    else:
        sigma = np.nanstd(y[noise_mask] - np.nanmean(y[noise_mask]))

    if sigma == 0 or np.isnan(sigma):
        print("Noise estimation failed!")
        return None

    # ---------- chi2 ----------
    n_params = 3 * len(comps)
    chi2 = np.sum(((y - total_model) / sigma) ** 2)
    dof = len(y) - n_params

    if dof > 0:
        chi2_red = chi2 / dof
    else:
        chi2_red = np.nan

    print(f"Reduced Chi² = {chi2_red:.3f}")
    print(f"Noise sigma = {sigma:.4f} K")
    print(f"Number of components = {len(comps)}")

    # =====================================================
    # Create aligned spectrum and residual panels.
    # =====================================================
    fig, (ax1, ax2) = plt.subplots(
        2,
        1,
        figsize=(8, 6),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1]},
    )

    # Upper panel: observed spectrum, total fit, and individual components.
    ax1.plot(
        x,
        y,
        "k-",
        linewidth=0.8,
        alpha=0.3,
        label="Original Data",
        zorder=1,
    )

    ax1.plot(
        x,
        total_model,
        linestyle="-",
        color="darkorange",
        linewidth=5,
        label=rf"Total Fit ($\chi^2_\nu$={chi2_red:.2f})",
        zorder=2,
    )

    for m in models:
        ax1.plot(
            x,
            m["model"],
            linestyle="-.",
            color=m["color"],
            linewidth=3,
            zorder=3,
        )

    # Separate component and data/model legends for readability.
    comp_legend_elements = [
        Line2D(
            [0],
            [0],
            color=m["color"],
            linestyle="-.",
            linewidth=3,
            label=m["label"],
        )
        for m in models
    ]

    legend1 = ax1.legend(
        handles=comp_legend_elements,
        loc="upper left",
        fontsize=16,
        framealpha=0.9,
        edgecolor="black",
        title_fontsize=12,
    )

    ax1.add_artist(legend1)

    data_legend_elements = [
        Line2D(
            [0],
            [0],
            color="black",
            linewidth=0.8,
            alpha=0.3,
            label="Original Data",
        ),
        Line2D(
            [0],
            [0],
            color="darkorange",
            linewidth=4,
            label=rf"Total Fit ($\chi^2_\nu$={chi2_red:.2f})",
        ),
    ]

    ax1.legend(
        handles=data_legend_elements,
        loc="upper right",
        fontsize=16,
        framealpha=0.9,
        edgecolor="black",
    )

    ax1.set_ylabel("Intensity [K]", fontsize=20)
    ax1.tick_params(axis="both", which="both", direction="out", labelsize=16)
    ax1.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    ax1.grid()
    ax1.set_xlim(*xlim)
    ax1.set_ylim(*ylim)

    # Lower panel: data-minus-model residual spectrum.
    residual = y - total_model

    ax2.plot(
        x,
        residual,
        "k-",
        linewidth=0.5,
        alpha=0.3,
    )

    ax2.axhline(
        0,
        linestyle="--",
        color="red",
        alpha=0.5,
    )

    ax2.tick_params(axis="both", which="both", direction="out", labelsize=18)
    ax2.set_xlabel("Velocity [km s$^{-1}$]", fontsize=20)
    ax2.set_ylabel("Residual [K]", fontsize=20)
    ax2.set_ylim(-0.4, 0.4)
    ax2.grid()

    # Save a source-specific or all-source PDF filename.
    if source_id is None:
        out_name = f"ROHSA_spectrum_i{i}_j{j}_all_sources.pdf"
    else:
        out_name = f"ROHSA_spectrum_i{i}_j{j}_source_{source_id:03d}.pdf"

    out_file = os.path.join(output_dir, out_name)

    plt.tight_layout()

    plt.savefig(
        out_file,
        bbox_inches="tight",
        dpi=300,
    )

    plt.show()

    print(f"Saved figure to: {out_file}")

    return chi2_red


# =========================================================
# 5. Configure and run the pixel-level fit diagnostic.
# =========================================================
if __name__ == "__main__":

    source_csv_dir = "./baseline/SNR=2/output_individual_source_10/output_merged_source_0.7/source_physical_parameters"

    pixel_results = read_merged_source_csv_to_rohsas_format(
        source_csv_dir=source_csv_dir,
    )

    # Select the target pixel using i=y and j=x.
    # i = y_pixel, j = x_pixel
    i, j = 79,16

    # Inspect available fitted components before plotting.
    print_available_components_at_pixel(
        pixel_results,
        i,
        j,
    )

    # Plot all final-source components present at the selected pixel.
    plot_with_residual_and_chi2(
        cube,
        pixel_results,
        i,
        j,
        source_id=None,
        output_dir="./baseline/SNR=2/output_individual_source_10/output_merged_source_0.7",
    )
